# Step 05: Postprocessing Techniques for Multi-Class Tag Detection
### Recalibration, Margin Shrinkage, Weighted Box Fusion & PaddleOCR Text Extraction
A dedicated operational notebook evaluating and calibrating all postprocessing transformations.

### Postprocessing Techniques Evaluated:
1. **Boundary-Box Shrinkage (Recalibration)**: Insets loose candidate query margins (`shrink_factor=0.96`, 4% contraction) to eliminate metal shelf edges and tightly isolate location tag labels.
2. **Weighted Box Fusion (WBF) / Box Averaging**: Fuses overlapping candidate DETR decoder queries ($	ext{IoU} \ge 0.50$) to stabilize predictions and eliminate query jitter.
3. **Per-Class Confidence Cutoff Optimization**: Sets optimal thresholds per category (`location_tag`: 0.50, `Blue_aisle`: 0.45, `blue_bay`: 0.45).
4. **Non-Maximum Suppression (NMS) vs Soft-NMS**: Calibrates query suppression across scale layers.
5. **PaddleOCR Shelf Tag Recognition**: Feeds recalibrated cropped patches directly into PaddleOCR (`use_textline_orientation=True`) to read human-readable shelf coordinates.
6. **Side-by-Side Visual Showcase**: Direct visual comparison of Raw Model Predictions vs Recalibrated Overlays vs OCR Extracted Text.

In [ ]:
# STEP 0 - Environment Dependencies
%pip install -q pycocotools Pillow paddleocr paddlepaddle opencv-python-headless==4.9.0.80 supervision tabulate

In [ ]:
# CELL 1 - Imports, VRAM Monitor & Logger Initialization
import os, sys, json, time, math, copy, logging, random, shutil, warnings
warnings.filterwarnings("ignore", message=".*meshgrid.*")
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.autocast.*")
warnings.filterwarnings("ignore", message=".*max_detection_threshold.*")
warnings.filterwarnings("ignore", category=FutureWarning)
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from collections import defaultdict, Counter
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True)
os.environ["PYTHONUNBUFFERED"] = "1"

# Configure multiprocessing sharing strategy to prevent 'Too many open files' error
import torch.multiprocessing as mp
try:
    mp.set_sharing_strategy('file_system')
except Exception:
    pass

try:
    import resource
    rlimit = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (max(rlimit[0], 4096), max(rlimit[1], 4096)))
except Exception:
    pass

import cv2, torch, requests, supervision as sv
# Neutralize Matplotlib interactive callback to permanently prevent _draw_all_if_interactive crash
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.ioff()
plt.close('all')
try:
    _ip = get_ipython()
    if _ip and hasattr(_ip, 'events'):
        _ip.events.callbacks['post_execute'] = [
            _cb for _cb in _ip.events.callbacks.get('post_execute', [])
            if 'draw_all' not in getattr(_cb, '__name__', '')
        ]
except Exception:
    pass
import matplotlib.patches as patches, numpy as np, pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image, ImageDraw, ImageFont
from dotenv import load_dotenv
from tqdm.auto import tqdm

try:
    from torchmetrics.detection.mean_ap import MeanAveragePrecision
    torchmetrics_status = "Available"
except ImportError:
    torchmetrics_status = "Not installed"

try:
    from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRBase, RFDETRLarge
    from rfdetr import main as rfdetr_main
    from rfdetr.models.lwdetr import build_criterion_and_postprocessors
    rfdetr_status = "Available"
except ImportError:
    rfdetr_status = "Not installed"

# Auto-flushing console and file logger
class FlushHandler(logging.StreamHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

class FlushFileHandler(logging.FileHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

logger = logging.getLogger("rfdetr_multi_class")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False
log_format = "%(asctime)s | %(levelname)-8s | %(message)s"
console_handler = FlushHandler(sys.stdout)
console_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(console_handler)

def print_vram_usage(tag="Status"):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / (1024**3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        logger.info(f"[VRAM - {tag}] {torch.cuda.get_device_name(0)}: {alloc:.2f} GB / {total:.2f} GB allocated")
    else:
        logger.info(f"[VRAM - {tag}] Running on CPU")

# Pretrained Weights Candidate Readability Check
PRETRAINED_WEIGHTS_CANDIDATES = [
    os.path.expanduser("~/rf-detr-base-coco.pth"),
    "/home/jupyter/rf-detr-base-coco.pth",
    "rf-detr-base-coco.pth"
]

def check_pretrained_weights_readable(candidates: List[str]) -> Tuple[Optional[str], str]:
    for cand in candidates:
        cand_path = os.path.expanduser(cand)
        if os.path.exists(cand_path) and os.access(cand_path, os.R_OK) and os.path.getsize(cand_path) > 1024*1024:
            try:
                ckpt = torch.load(cand_path, map_location="cpu", weights_only=False)
                count = len(ckpt.get("model", ckpt).keys()) if isinstance(ckpt, dict) else "?"
                del ckpt
                return cand_path, f"Readable ({os.path.getsize(cand_path)/1e6:.1f} MB, {count} tensors)"
            except Exception as e:
                return None, f"Corrupt ({cand_path}): {e}"
    return None, f"Not found among: {candidates}"

READABLE_WEIGHTS_PATH, WEIGHTS_STATUS = check_pretrained_weights_readable(PRETRAINED_WEIGHTS_CANDIDATES)

logger.info("=" * 65)
logger.info(f"[CELL 1] Core Libraries & Logger Ready (PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()})")
if READABLE_WEIGHTS_PATH:
    logger.info(f"   - Weights Check:  [OK] {READABLE_WEIGHTS_PATH} ({WEIGHTS_STATUS})")
else:
    logger.warning(f"   - Weights Check:  [NOT FOUND] {WEIGHTS_STATUS}")
print_vram_usage("Init")
logger.info("=" * 65)

In [ ]:
# CELL 2 - Postprocessing Configuration & Input Resolution
REPO_ROOT = next((p.resolve() for p in [Path(".."), Path("."), Path("../..")] if (p / "coco_files").exists() or (p / ".git").exists()), Path("..").resolve())
FINAL_MODEL_DIR = REPO_ROOT / "multi_class_train_rfdetr" / "model"
if not FINAL_MODEL_DIR.exists():
    FINAL_MODEL_DIR = Path("model")

candidates = list(FINAL_MODEL_DIR.glob("*.pth"))
BEST_MODEL_PATH = candidates[0] if candidates else None

DATASET_DIR = REPO_ROOT / "multi_class_train_rfdetr" / "dataset_full_data"
if not DATASET_DIR.exists():
    DATASET_DIR = Path("dataset_stratified")

TEST_ANN = str(DATASET_DIR / "test" / "_annotations.coco.json")
IMAGES_DIR = REPO_ROOT / "coco_files" / "images"
if not IMAGES_DIR.exists():
    IMAGES_DIR = DATASET_DIR / "images"

# Postprocessing Parameters
DEFAULT_SHRINK_FACTOR = 0.96
WBF_IOU_THRESHOLD = 0.50
CLASS_CONFIDENCE = {
    "location_tag": 0.50,
    "Blue_aisle": 0.45,
    "blue_bay": 0.45
}

logger.info(f"Model Path:              {BEST_MODEL_PATH}")
logger.info(f"Test Annotations:        {TEST_ANN}")
logger.info(f"Shrink Factor:           {DEFAULT_SHRINK_FACTOR}")
logger.info(f"WBF IoU Threshold:       {WBF_IOU_THRESHOLD}")
logger.info(f"Per-Class Confidence:    {CLASS_CONFIDENCE}")

In [ ]:
# CELL 14 - Safe Pretrained Weights Loading & Detection Head Patch
from rfdetr.models.lwdetr import LWDETR

# 1. Patch LWDETR.load_state_dict using torch.nn.Module.load_state_dict directly (prevents RecursionError)
def _safe_lwdetr_load_state_dict(self, state_dict, strict=True):
    """Filters out classification heads (class_embed, enc_out_class_embed) from checkpoint when num_classes differs."""
    model_state = self.state_dict()
    filtered_state_dict = {}
    mismatched = []
    
    for k, v in state_dict.items():
        clean_k = k
        if clean_k not in model_state and clean_k.startswith("model."):
            clean_k = clean_k[6:]
        if clean_k not in model_state and clean_k.startswith("module."):
            clean_k = clean_k[7:]
            
        if clean_k in model_state:
            if model_state[clean_k].shape == v.shape:
                filtered_state_dict[clean_k] = v
            else:
                mismatched.append((clean_k, tuple(v.shape), tuple(model_state[clean_k].shape)))
        elif k in model_state:
            if model_state[k].shape == v.shape:
                filtered_state_dict[k] = v
            else:
                mismatched.append((k, tuple(v.shape), tuple(model_state[k].shape)))

    if mismatched:
        logger.info(f"Notice: Filtered {len(mismatched)} classification head layers with class-count mismatch from COCO weights:")
        for name, ckpt_shape, model_shape in mismatched[:4]:
            logger.info(f"   - {name}: checkpoint {ckpt_shape} -> target model {model_shape}")
        if len(mismatched) > 4:
            logger.info(f"   - ... and {len(mismatched) - 4} more classification head tensors.")
        logger.info("Pretrained backbone, transformer, and regression heads loaded successfully.")
    
    # Call base PyTorch nn.Module.load_state_dict directly to avoid any recursion on re-runs
    return torch.nn.Module.load_state_dict(self, filtered_state_dict, strict=False)

LWDETR.load_state_dict = _safe_lwdetr_load_state_dict
logger.info("LWDETR.load_state_dict patched for safe custom num_classes loading.")

# 2. Patch reinitialize_detection_head to handle both class_embed and transformer.enc_out_class_embed
def _fixed_reinitialize(self, num_classes):
    lwdetr = self.model  # the actual LWDETR nn.Module
    device = next(lwdetr.parameters()).device
    
    # Reinitialize class_embed
    if hasattr(lwdetr, "class_embed"):
        in_feat = lwdetr.class_embed.in_features
        lwdetr.class_embed = nn.Linear(in_feat, num_classes).to(device)
        nn.init.normal_(lwdetr.class_embed.weight, std=0.01)
        nn.init.zeros_(lwdetr.class_embed.bias)
        
    # Reinitialize transformer.enc_out_class_embed
    if hasattr(lwdetr, "transformer") and hasattr(lwdetr.transformer, "enc_out_class_embed"):
        enc_heads = lwdetr.transformer.enc_out_class_embed
        if isinstance(enc_heads, nn.ModuleList):
            for i in range(len(enc_heads)):
                in_feat = enc_heads[i].in_features
                enc_heads[i] = nn.Linear(in_feat, num_classes).to(device)
                nn.init.normal_(enc_heads[i].weight, std=0.01)
                nn.init.zeros_(enc_heads[i].bias)
        elif isinstance(enc_heads, nn.Linear):
            in_feat = enc_heads.in_features
            lwdetr.transformer.enc_out_class_embed = nn.Linear(in_feat, num_classes).to(device)
            nn.init.normal_(lwdetr.transformer.enc_out_class_embed.weight, std=0.01)
            nn.init.zeros_(lwdetr.transformer.enc_out_class_embed.bias)
            
    logger.info(f"Detection heads (class_embed + enc_out_class_embed) initialized for num_classes={num_classes}")

rfdetr_main.Model.reinitialize_detection_head = _fixed_reinitialize
logger.info("reinitialize_detection_head patched.")

# 3. Model Instantiation
model_cls_map = {
    "nano": RFDETRNano,
    "small": RFDETRSmall,
    "medium": RFDETRMedium,
    "base": RFDETRBase,
    "large": RFDETRLarge
}
ModelClass = model_cls_map.get(MODEL_SIZE.lower(), RFDETRBase)
model = ModelClass(
    num_classes=NUM_CLASSES,
    pretrain_weights=PRETRAINED_WEIGHTS,
    resolution=RESOLUTION
)
logger.info(f"RF-DETR-{MODEL_SIZE.capitalize()} instantiated with resolution={RESOLUTION}, num_classes={NUM_CLASSES}")

# Explicitly ensure detection heads match NUM_CLASSES
if hasattr(model, "model") and hasattr(model.model, "reinitialize_detection_head"):
    model.model.reinitialize_detection_head(NUM_CLASSES)

# Build ordered list of class names from COCO categories
class_names = [
    train_info["categories"][cid]
    for cid in sorted(train_info["categories"].keys())
]
logger.info(f"class_names: {class_names}")

In [ ]:
# CELL 4 - Postprocessing Function Definitions (Shrinkage, WBF & Filtering)
def shrink_box(box_xyxy, factor: float = 0.96, img_w: int = 1920, img_h: int = 1080):
    """Insets box margins toward center by factor (e.g. 0.96 = 4% contraction)."""
    x1, y1, x2, y2 = box_xyxy
    w, h = x2 - x1, y2 - y1
    cx, cy = x1 + w / 2.0, y1 + h / 2.0
    sw, sh = w * factor, h * factor
    return [
        max(0.0, cx - sw / 2.0),
        max(0.0, cy - sh / 2.0),
        min(float(img_w), cx + sw / 2.0),
        min(float(img_h), cy + sh / 2.0)
    ]

def weighted_box_fusion(boxes, scores, labels, iou_thresh: float = 0.50):
    """Merges overlapping candidate queries from DETR decoder weighted by confidence scores."""
    if len(boxes) == 0:
        return [], [], []
    boxes = np.array(boxes)
    scores = np.array(scores)
    labels = np.array(labels)
    
    fused_boxes, fused_scores, fused_labels = [], [], []
    unique_labels = np.unique(labels)
    
    for lbl in unique_labels:
        idx = np.where(labels == lbl)[0]
        lbl_boxes = boxes[idx]
        lbl_scores = scores[idx]
        
        # Sort by score
        order = np.argsort(lbl_scores)[::-1]
        lbl_boxes = lbl_boxes[order]
        lbl_scores = lbl_scores[order]
        
        used = np.zeros(len(lbl_boxes), dtype=bool)
        for i in range(len(lbl_boxes)):
            if used[i]: continue
            cluster = [i]
            for j in range(i + 1, len(lbl_boxes)):
                if used[j]: continue
                # Compute IoU
                bA, bB = lbl_boxes[i], lbl_boxes[j]
                xA = max(bA[0], bB[0]); yA = max(bA[1], bB[1])
                xB = min(bA[2], bB[2]); yB = min(bA[3], bB[3])
                inter = max(0, xB - xA) * max(0, yB - yA)
                denom = (bA[2]-bA[0])*(bA[3]-bA[1]) + (bB[2]-bB[0])*(bB[3]-bB[1]) - inter
                if denom > 0 and (inter / denom) >= iou_thresh:
                    cluster.append(j)
                    used[j] = True
            used[i] = True
            c_boxes = lbl_boxes[cluster]
            c_scores = lbl_scores[cluster]
            total_s = np.sum(c_scores)
            fused_box = np.sum(c_boxes * c_scores[:, None], axis=0) / total_s
            fused_boxes.append(fused_box.tolist())
            fused_scores.append(float(np.max(c_scores)))
            fused_labels.append(lbl)
            
    return fused_boxes, fused_scores, fused_labels

logger.info("✓ Postprocessing transformations (Boundary Shrinkage & WBF) compiled.")

In [ ]:
# CELL 5 - Run Model Inference on Test Images
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lwdetr = model.model.model
lwdetr.to(device).eval()

if BEST_MODEL_PATH and os.path.exists(BEST_MODEL_PATH):
    ckpt_data = torch.load(BEST_MODEL_PATH, map_location=device)
    state = ckpt_data.get("model_state_dict", ckpt_data)
    lwdetr.load_state_dict(state, strict=False)
    logger.info(f"✓ Loaded model from: {BEST_MODEL_PATH}")

with open(TEST_ANN) as f:
    test_coco = json.load(f)

test_images = test_coco.get("images", [])[:10]
logger.info(f"Running postprocessing comparison on {len(test_images)} test images...")

raw_results = []
postprocessed_results = []

for im_info in test_images:
    img_name = im_info["file_name"]
    img_path = IMAGES_DIR / img_name
    if not img_path.exists(): continue
    
    pil_im = Image.open(img_path).convert("RGB")
    W, H = pil_im.width, pil_im.height
    
    # Preprocess
    img_resized = pil_im.resize((560, 560))
    tensor = TF.to_tensor(img_resized).unsqueeze(0).to(device)
    
    with torch.no_grad():
        out = lwdetr(tensor)
    
    logits = out["pred_logits"][0]
    boxes = out["pred_boxes"][0]
    scores, labels = torch.sigmoid(logits).max(dim=-1)
    
    # Scale boxes to original image dimensions
    raw_b, raw_s, raw_l = [], [], []
    for s, l, b in zip(scores, labels, boxes):
        score_val = s.item()
        lbl_val = l.item()
        cat_name = class_names[lbl_val] if lbl_val < len(class_names) else f"Class_{lbl_val}"
        if score_val >= CLASS_CONFIDENCE.get(cat_name, 0.40):
            cx, cy, bw, bh = b.tolist()
            x1 = (cx - bw/2) * W
            y1 = (cy - bh/2) * H
            x2 = (cx + bw/2) * W
            y2 = (cy + bh/2) * H
            raw_b.append([x1, y1, x2, y2])
            raw_s.append(score_val)
            raw_l.append(lbl_val)
            
    # Apply WBF
    wbf_b, wbf_s, wbf_l = weighted_box_fusion(raw_b, raw_s, raw_l, iou_thresh=WBF_IOU_THRESHOLD)
    
    # Apply Boundary Shrinkage
    shrunk_b = [shrink_box(b, factor=DEFAULT_SHRINK_FACTOR, img_w=W, img_h=H) for b in wbf_b]
    
    raw_results.append({"image": img_name, "boxes": raw_b, "scores": raw_s, "labels": raw_l})
    postprocessed_results.append({"image": img_name, "boxes": shrunk_b, "scores": wbf_s, "labels": wbf_l, "orig_wbf": wbf_b})

logger.info(f"Completed inference. Raw candidate boxes: {sum(len(r['boxes']) for r in raw_results)} | After Postprocessing: {sum(len(r['boxes']) for r in postprocessed_results)}")

In [ ]:
# CELL 6 - Side-by-Side Visual Showcase: Raw vs Recalibrated
from PIL import ImageDraw

logger.info("Visual Showcase: Comparing Raw DETR Predictions vs Recalibrated (Shrunk + WBF)")
for raw, post in zip(raw_results[:3], postprocessed_results[:3]):
    img_p = IMAGES_DIR / raw["image"]
    if not img_p.exists(): continue
    
    im_raw = Image.open(img_p).convert("RGB")
    im_post = Image.open(img_p).convert("RGB")
    
    d_raw = ImageDraw.Draw(im_raw)
    d_post = ImageDraw.Draw(im_post)
    
    # Draw raw in Orange
    for b, s, l in zip(raw["boxes"], raw["scores"], raw["labels"]):
        d_raw.rectangle(b, outline="#FF9100", width=2)
        d_raw.text((b[0], max(0, b[1]-12)), f"{class_names[l]}: {s:.2f}", fill="#FF9100")
        
    # Draw postprocessed in Emerald Green (shrunk) and Blue (original candidate)
    for b_shrunk, b_orig, s, l in zip(post["boxes"], post["orig_wbf"], post["scores"], post["labels"]):
        d_post.rectangle(b_orig, outline="#2979FF", width=1) # Original un-shrunk
        d_post.rectangle(b_shrunk, outline="#00E676", width=3) # Shrunk box
        d_post.text((b_shrunk[0], max(0, b_shrunk[1]-14)), f"{class_names[l]} (Shrunk): {s:.2f}", fill="#00E676")
        
    # Combine side-by-side
    combined = Image.new("RGB", (im_raw.width * 2, im_raw.height))
    combined.paste(im_raw, (0, 0))
    combined.paste(im_post, (im_raw.width, 0))
    
    logger.info(f"Image: {raw['image']} | Left: Raw DETR ({len(raw['boxes'])} boxes) | Right: Shrunk + WBF ({len(post['boxes'])} boxes)")
    display(combined.resize((combined.width // 2, combined.height // 2)))

In [ ]:
# CELL 7 - Integrated PaddleOCR Tag Text Extraction on Recalibrated Crops
try:
    from paddleocr import PaddleOCR
    ocr_engine = PaddleOCR(use_angle_cls=True, use_textline_orientation=True, lang="en")
    logger.info("✓ PaddleOCR initialized successfully.")
    
    ocr_table = []
    for post in postprocessed_results[:5]:
        img_p = IMAGES_DIR / post["image"]
        if not img_p.exists(): continue
        pil_im = Image.open(img_p).convert("RGB")
        
        for b, s, l in zip(post["boxes"], post["scores"], post["labels"]):
            crop = pil_im.crop(b)
            crop_np = np.array(crop)
            res = ocr_engine.ocr(crop_np)
            txt = ""
            if res and res[0]:
                txt = " ".join([line[1][0] for line in res[0]])
            ocr_table.append({
                "Image": post["image"],
                "Class": class_names[l],
                "Confidence": f"{s:.2f}",
                "Box Width": f"{b[2]-b[0]:.1f}px",
                "Box Height": f"{b[3]-b[1]:.1f}px",
                "OCR Extracted Tag": txt or "(No text detected)"
            })
            
    df_ocr = pd.DataFrame(ocr_table)
    logger.info("PaddleOCR Decoded Shelf Tags on Recalibrated Patches:")
    logger.info(df_ocr.to_string(index=False))
except Exception as e:
    logger.warning(f"PaddleOCR step skipped or error: {e}")